In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from summaries import Analyzer, Cleaner
import html
import glob
import re
import emoji
from pandarallel import pandarallel
from sympy import true

%matplotlib inline

# Load and clean dataset

In [2]:
DATA_PATH = "/home/jacob/release/"
TRAIN_FILE = "train.jsonl.gz"
TEST_FILE = "test.jsonl.gz"
VAL_FILE = "dev.jsonl.gz"

In [ ]:
#chunk dataset, remove ine breaks and html
for file in [TRAIN_FILE, TEST_FILE, VAL_FILE]:
    reader = pd.read_json(DATA_PATH + file, lines=True, compression='gzip', chunksize=200000)
    chunk = 0
    for df in reader:
        df = df.drop(columns=['url', 'archive', 'title', 'date', 'compression',
                               'coverage', 'density', 'compression_bin', 'coverage_bin',
                               'density_bin'])
        print("loaded")
        df["text"] = df["text"].str.replace("\n", " ", regex=False)
        df["summary"] = df["summary"].str.replace("\n", " ", regex=False)

        df["summary"] = df["summary"].apply(html.unescape)
        df["text"] = df["text"].apply(html.unescape)
        print("cleaned")

        df.to_csv(f'{DATA_PATH + "filtered/" + file.partition(".")[0]}_{chunk}.csv', index=False)
        print("Saved ", f'{file.partition(".")[0]}_{chunk}.csv')
        chunk += 1

In [3]:
def clean_text(text: str) -> str:

    # 3) Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # 4) Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)

    # 5) Remove emojis completely
    text = emoji.replace_emoji(text, replace="")

    # 6) Remove special characters (keep letters, numbers, spaces)
    #text = re.sub(r"[^a-z0-9\s]", " ", text)

    # 7) Reduce repeated characters
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # 8) Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
train_files = glob.glob(DATA_PATH + "unfiltered/" + "train_*.csv")
test_files = glob.glob(DATA_PATH + "unfiltered/" + "test_*.csv")
dev_files = glob.glob(DATA_PATH + "unfiltered/" + "dev_*.csv")

for files in [train_files, test_files, dev_files]:
    df = pd.concat(
        (pd.read_csv(file) for file in files),
        ignore_index=True
    )

    pandarallel.initialize(progress_bar=True, use_memory_fs=False, nb_workers=5)
    df["text"] = df["text"].parallel_apply(clean_text)
    df["summary"] = df["summary"].parallel_apply(clean_text)

    df.to_csv(files[0].replace("unfiltered", "filtered"))




INFO: Pandarallel will run on 5 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [ ]:
set = []
for text in df["text"]:
    for char in text:
        if not char.isalnum() and not char in set:
            print(char)
            set.append(char)
for text in df["summary"]:
    for char in text:
        if not char.isalnum() and not char in set:
            print(char)
            set.append(char)

In [ ]:
# Apply cleaner to filter dataset by text length, summary length, and compression ratio
for

analyzer = Analyzer(lemmatize=True, lang="en")
cleaner = Cleaner(analyzer, min_length_summary = 18, min_length_reference = 250, length_metric = "whitespace")

df = cleaner.clean_dataset("summary", "text", df.to_dict(), )

In [ ]:
text_lengths = df['text'].str.len()
sum_lengths = df['summary'].str.len()

print("Text Summary:")
print(text_lengths.describe())
print("\nSummary Summary:")
print(sum_lengths.describe())

In [ ]:
sns.histplot(text_lengths, kde=False, stat="percent", binwidth=1000)
plt.xlim(0, 100000)
plt.show()

num_10000 = sum(1 for i in text_lengths if i<10000)
num_20000 = sum(1 for i in text_lengths if i<20000)
total = len(text_lengths)

print("Percent of text under 10000:", num_10000 / total)
print("Percent of text under 20000:", num_20000 / total)

sns.histplot(df['summary'].str.len(), kde=False, stat="percent", binwidth=100)
plt.xlim(0, 1000)
plt.show()